{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "19e2a8c0",
   "metadata": {},
   "source": [
    "# Peramalan kadar $NO_2$ di daerah Bangkalan Madura (revisi)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "9c5126d3",
   "metadata": {},
   "source": [
    "## Latar Belakang\n",
    "\n",
    "Peningkatan aktivitas industri, transportasi, serta pertumbuhan populasi yang pesat telah menyebabkan peningkatan signifikan terhadap tingkat pencemaran udara di berbagai wilayah. Salah satu polutan udara utama yang menjadi perhatian adalah Nitrogen Dioksida (NO₂), yaitu gas beracun yang dihasilkan terutama dari proses pembakaran bahan bakar fosil seperti kendaraan bermotor, pembangkit listrik, dan kegiatan industri. NO₂ memiliki dampak serius terhadap kesehatan manusia, seperti gangguan pernapasan, iritasi paru-paru, serta memperburuk penyakit asma dan bronkitis. Selain itu, NO₂ juga berkontribusi terhadap pembentukan hujan asam dan penurunan kualitas lingkungan secara keseluruhan.\n",
    "\n",
    "## 1. Pengumpulan Data\n",
    "\n",
    "Pertama kita akan mengumpulkan data Time Series Harian kadar NO2 di daerah Bangkalan. Pengumpulan data dari sumber website https://dataspace.copernicus.eu/ , buat akun terlebih dahulu di website copernicus tersebut. \n",
    "\n",
    "Dokumentasi cara pengambilan data di https://documentation.dataspace.copernicus.eu/notebook-samples/openeo/NO2Covid.html .\n",
    "\n",
    "Untuk menuliskan code Python untuk mengambil data, silahkan kunjungi halaman https://dataspace.copernicus.eu/analyse/jupyterlab, klik Access JupyterLab, scroll kebawah sedikit ..., lalu pilih Python 3 (ipykernel)\n",
    "\n",
    "![Teks alternatif](img/Screenshot%202025-10-23%20105545.png)\n",
    "\n",
    "Disini kita akan mengambil data kadar NO2 di daerah Bangkalan dari tanggal ... sampai ... .\n",
    "\n",
    "Kita install terlebih dahulu openoneo:\n",
    "```{code}\n",
    "pip install openeo\n",
    "```\n",
    "\n",
    "Lalu tuliskan code dibawah:\n",
    "\n",
    "```{code}\n",
    "import openeo\n",
    "```\n",
    "\n",
    "```{code}\n",
    "connection = openeo.connect(\"openeo.dataspace.copernicus.eu\").authenticate_oidc()\n",
    "```\n",
    "\n",
    "pada saat menjalankan baris code diatas (connection), nanti akan diminta authentikasi seperti output berikut:\n",
    "\n",
    "```{code}\n",
    "Terminal/Output\n",
    "Visit (link authentikasi) 📋 to authenticate.\n",
    "✅ Authorized successfully\n",
    "Authenticated using device code flow.\n",
    "```\n",
    "\n",
    "Kalian tinggal klik link authentikasi lalu login menggunakan akun \"copernicus\" kalian.\n",
    "\n",
    "```{code}\n",
    "aoi = {\n",
    "    \"type\": \"Polygon\",\n",
    "    \"coordinates\": [\n",
    "        [\n",
    "            [113.09, -6.89],\n",
    "            [112.68, -6.89],\n",
    "            [112.68, -7.20],\n",
    "            [113.09, -7.20],\n",
    "            [113.09, -6.89],\n",
    "        ]\n",
    "    ]\n",
    "}\n",
    "\n",
    "s5post = connection.load_collection(\n",
    "    \"SENTINEL_5P_L2\",\n",
    "    temporal_extent=[\"2023-10-01\", \"2025-10-01\"],\n",
    "    spatial_extent={\n",
    "        \"west\": 112.68,\n",
    "        \"south\": -7.20,\n",
    "        \"east\": 113.09,\n",
    "        \"north\": -6.89\n",
    "    },\n",
    "    bands=[\"NO2\"],\n",
    ")\n",
    "\n",
    "# Now aggregate by day to avoid having multiple data per day\n",
    "s5p_no2_daily = s5post.aggregate_temporal_period(reducer=\"mean\", period=\"day\")\n",
    "\n",
    "# Now create a spatial aggregation to generate mean timeseries data\n",
    "s5p_no2_aoi = s5p_no2_daily.aggregate_spatial(reducer=\"mean\", geometries=aoi)\n",
    "```\n",
    "\n",
    "Code diatas memerlukan titik koordinasi area yang akan diambil data $NO_2$-nya, untuk mengambil titik koordinasi kaian kunjungi webiste https://geojson.io/#map=14.8/-7.04732/112.69463 . Didalam website tersebut kalian akan memilih daerah dengan cara memberi shape kotak didaerah yang ingin kalian ambil datanya.\n",
    "\n",
    "![Teks alternatif](img/Screenshot%202025-10-23%20110952.png)\n",
    "\n",
    "Di panel sebelah kanan terdapat data JSON yang berupa koordinat daerah yang kalian pilih, kalian salin terus sesuaikan dengan code diatas di bagian variabel \"aoi\" dan spatial_extent."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "7ebe684a",
   "metadata": {},
   "source": [
    "Lalu kalian tambahkan baris code dibawah untuk memulai pengambilan data:\n",
    "\n",
    "```{code}\n",
    "job = s5post.execute_batch(title=\"NO2 in Bangkalan\", outputfile=\"NO2Bangkalan.nc\")\n",
    "```\n",
    "\n",
    "Tunggu proses pengambilan data, output proses seperti berikut:\n",
    "\n",
    "```{code}\n",
    "0:00:00 Job 'j-2510231608434524a87dedeacfaf5a43': send 'start'\n",
    "0:00:15 Job 'j-2510231608434524a87dedeacfaf5a43': created (progress 0%)\n",
    "0:00:20 Job 'j-2510231608434524a87dedeacfaf5a43': created (progress 0%)\n",
    "0:00:26 Job 'j-2510231608434524a87dedeacfaf5a43': created (progress 0%)\n",
    "0:00:35 Job 'j-2510231608434524a87dedeacfaf5a43': queued (progress 0%)\n",
    "0:00:46 Job 'j-2510231608434524a87dedeacfaf5a43': queued (progress 0%)\n",
    "0:00:58 Job 'j-2510231608434524a87dedeacfaf5a43': queued (progress 0%)\n",
    "0:01:14 Job 'j-2510231608434524a87dedeacfaf5a43': queued (progress 0%)\n",
    "0:01:33 Job 'j-2510231608434524a87dedeacfaf5a43': running (progress N/A)\n",
    "0:01:57 Job 'j-2510231608434524a87dedeacfaf5a43': running (progress N/A)\n",
    "0:02:27 Job 'j-2510231608434524a87dedeacfaf5a43': running (progress N/A)\n",
    "0:03:05 Job 'j-2510231608434524a87dedeacfaf5a43': running (progress N/A)\n",
    "0:03:52 Job 'j-2510231608434524a87dedeacfaf5a43': running (progress N/A)\n",
    "0:04:50 Job 'j-2510231608434524a87dedeacfaf5a43': running (progress N/A)\n",
    "0:05:50 Job 'j-2510231608434524a87dedeacfaf5a43': running (progress N/A)\n",
    "0:06:50 Job 'j-2510231608434524a87dedeacfaf5a43': running (progress N/A)\n",
    "0:07:50 Job 'j-2510231608434524a87dedeacfaf5a43': finished (progress 100%)\n",
    "```\n",
    "\n",
    "Abaikan ketika ada N/A.\n",
    "\n",
    "Ketika proses pengambilan data, aktivitas kalian akan terekam di halaman https://editor.openeo.org/?server=https%3A%2F%2Fopeneo.dataspace.copernicus.eu%2Fopeneo%2F1.2 . Disitu terdapat nama dataset dan status pengambilan data.\n",
    "\n",
    "![Teks alternatif](img/Screenshot%202025-10-24%20121430.png)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "1e669820",
   "metadata": {},
   "source": [
    "## 2. Preproccessing Data\n",
    "\n",
    "Setelah kita mengambil data, data bisa diunduh di halaman https://editor.openeo.org/?server=https%3A%2F%2Fopeneo.dataspace.copernicus.eu%2Fopeneo%2F1.2 . File akan berbentuk .nc. Kita cuman perlu kolom date dan NO2 menggunakan code dibawah:\n",
    "\n",
    "```{code}\n",
    "import netCDF4\n",
    "\n",
    "file_path = \"data/NO2Bangkalan.nc\"\n",
    "ds = netCDF4.Dataset(file_path)\n",
    "\n",
    "# Lihat seluruh variabel yang tersedia\n",
    "print(\"📦 Variabel dalam file:\")\n",
    "print(ds.variables.keys())\n",
    "# dict_keys(['t', 'x', 'y', 'crs', 'NO2'])\n",
    "\n",
    "# Ambil NO2\n",
    "no2 = ds.variables[\"NO2\"][:]\n",
    "\n",
    "# Ambil Time\n",
    "time = ds.variables[\"t\"][:]\n",
    "\n",
    "# Konversi waktu ke format tanggal jika punya atribut 'units'\n",
    "try:\n",
    "    time_units = ds.variables[\"t\"].units\n",
    "    dates = netCDF4.num2date(time, units=time_units)\n",
    "except Exception:\n",
    "    dates = time  # fallback kalau tidak ada units\n",
    "\n",
    "# Tampilkan struktur data NO2\n",
    "print(type(no2))\n",
    "# type <class 'numpy.ma.core.MaskedArray'>\n",
    "\n",
    "print(len(no2))\n",
    "# banyaknya data record NO2 725\n",
    "\n",
    "print(len(no2[0]))\n",
    "# panjang data perbaris 9\n",
    "\n",
    "print(len(no2[0][0]))\n",
    "# panjang perdata 8\n",
    "\n",
    "print(no2[0][0][0])\n",
    "# 3.7701793e-05\n",
    "```\n",
    "\n",
    "Dari code diatas kita mengetahui bentuk data dari kolom NO2 nya.\n",
    "\n",
    "jadi struktur data NO2 perbaris adalah:\n",
    "\n",
    "```{code}\n",
    "[\n",
    "    [[] * 8] * 9\n",
    "]\n",
    "```\n",
    "\n",
    "Untuk melihat 10 data pertama adalah:\n",
    "\n",
    "```{code}\n",
    "print(\"Contoh data pertama:\")\n",
    "for i in range(0, 10):\n",
    "    print(no2[i])\n",
    "```\n",
    "\n",
    "Dalam sehari, terdapat banyak data NO2, jadi kita rata-ratakan agar satu cell data hanya terdapat satu value. Namun terdapat masalah pada data NO2 seperti missing value. Contoh pada output dibawah:\n",
    "\n",
    "```{code}\n",
    "Terminal/Output\n",
    "[2.9651806471520104e-05 4.1052295273402706e-05 -- 5.6563803809694946e-05\n",
    " -- -- 6.348737952066585e-05 --]\n",
    "```\n",
    "\n",
    "### a. Mengatasi Missing Value menggunakan metode Interpolasi Linear\n",
    "\n",
    "Sekarang kita akan mengatasi permasalahan missing value pada data NO2.\n",
    "\n",
    "```{code}\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "\n",
    "# Interpolasi Linear\n",
    "no2_filled = np.zeros_like(no2)\n",
    "# Untuk jaga-jaga jika terdapat '--' tidak berubah menjadi 0\n",
    "no2_filled = no2_filled.filled(0)\n",
    "\n",
    "# loop tiap grid (y,x)\n",
    "for i in range(no2.shape[1]):     # 9 baris\n",
    "    for j in range(no2.shape[2]): # 8 kolom\n",
    "        series = pd.Series(no2[:, i, j])\n",
    "        no2_filled[:, i, j] = series.interpolate(method='linear', limit_direction='both').to_numpy()\n",
    "```\n",
    "\n",
    "Dengan code diatas, missing value yang terdapat pada data NO2 akan diisi secara otomatis menggunakan metode Interpolasi Linear.\n",
    "\n",
    "### b. Rata-rata kan Data dan ubah Datetime\n",
    "\n",
    "Setelah mengatasi missing value, kita akan me-rata-rata-kan data NO2 agar satu record hanya berupa single value. Sekalian kita mengambil date nya dan menaruh di array. Kita akan mengubah datetime dari awalnya (2023-10-04 00:00:00) menjadi (2023-10-04) karena kita mengambil data time series harian jadi kita tidak memerlukan data jam, menit dan detik.\n",
    "\n",
    "```{code}\n",
    "new_dates = []\n",
    "new_no2 = []\n",
    "for i in range(len(dates)):\n",
    "    # ubah format datetime\n",
    "    new_date = dates[i].strftime('%Y-%m-%d')\n",
    "    new_dates.append(new_date)\n",
    "    new_no2.append(np.mean(no2_filled[i]))\n",
    "```\n",
    "\n",
    "### c. Simpan data dalam bentuk CSV\n",
    "\n",
    "Setelah itu kita akan membentuk data menjadi DataFrame Pandas untuk disimpan menjadi CSV.\n",
    "\n",
    "```{code}\n",
    "df = pd.DataFrame({\n",
    "    \"date\": dates,\n",
    "    \"NO2\": no2_values\n",
    "})\n",
    "\n",
    "# Simpan ke CSV\n",
    "df.to_csv(\"NO2_Bangkalan_timeseries.csv\", index=False)\n",
    "```\n",
    "\n",
    "Untuk mengatasi missing value dan menyimpan data ke CSV sudah berhasil."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "2dc07b6b",
   "metadata": {},
   "source": [
    "### d. Pengecekan Missing Value data harian pada CSV\n",
    "\n",
    "Sekarang setelah data berbentuk CSV, kita cek apakah data Time Series harian lengkap. Cara men-cek apakah data Time Series Harian lengkap gunakan code dibawah:\n",
    "\n",
    "```{code}\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "\n",
    "df = pd.read_csv(\"NO2_Bangkalan_timeseries.csv\")\n",
    "\n",
    "# Pastikan kolom 'date' bertipe datetime\n",
    "df['date'] = pd.to_datetime(df['date'])\n",
    "\n",
    "# Buat rentang tanggal lengkap\n",
    "start_date = \"2023-10-01\"\n",
    "end_date = \"2025-09-30\"\n",
    "full_range = pd.date_range(start=start_date, end=end_date, freq='D')\n",
    "\n",
    "# Cek tanggal yang hilang\n",
    "missing_dates = full_range.difference(df['date'])\n",
    "\n",
    "print(f\"Jumlah hari missing: {len(missing_dates)}\")\n",
    "print(\"Daftar tanggal missing:\")\n",
    "print(missing_dates)\n",
    "```\n",
    "\n",
    "```{code}\n",
    "output/terminal\n",
    "Jumlah hari missing: 6\n",
    "Daftar tanggal missing:\n",
    "DatetimeIndex(['2023-11-11', '2024-01-01', '2024-03-23', '2024-08-12',\n",
    "               '2025-01-30', '2025-01-31'],\n",
    "              dtype='datetime64[ns]', freq=None)\n",
    "```\n",
    "\n",
    "Dalam kasus saya ini, terdapat 6 hari missing value. Kita akan mengatasi lagi missing value menggunakan metode Interpolasi Linear. Cara memperbaikinya gunakan code dibawah:\n",
    "\n",
    "```{code}\n",
    "import pandas as pd\n",
    "\n",
    "# Pastikan datetime dan sorting\n",
    "df['date'] = pd.to_datetime(df['date'])\n",
    "df = df.sort_values('date')\n",
    "\n",
    "# Buat rentang tanggal lengkap\n",
    "full_range = pd.date_range(start=\"2023-10-01\", end=\"2025-09-30\", freq='D')\n",
    "\n",
    "# Reindex agar tanggal yang hilang muncul sebagai NaN\n",
    "df = df.set_index('date').reindex(full_range)\n",
    "df.index.name = 'date'\n",
    "\n",
    "# Interpolasi linear berdasarkan indeks waktu\n",
    "df['NO2'] = df['NO2'].interpolate(method='time')\n",
    "\n",
    "# (Opsional) jika masih ada NaN di bagian awal/akhir bisa gunakan forward/backward fill\n",
    "df['NO2'] = df['NO2'].fillna(method='bfill').fillna(method='ffill')\n",
    "\n",
    "# Simpan kembali ke CSV\n",
    "df.to_csv(\"no2_timeseries_interpolated.csv\")\n",
    "```\n",
    "\n",
    "Setelah saya cek missing value harian, sudah tidak ada lagi missing value.\n",
    "\n",
    "```{code}\n",
    "Jumlah hari missing: 0\n",
    "Daftar tanggal missing:\n",
    "DatetimeIndex([], dtype='datetime64[ns]', freq='D')\n",
    "```\n",
    "\n",
    "dengan bentuk data terdapat 2 kolom, kolom pertama yaitu date atau tanggal, kolom kedua yaitu kadar NO2 yang sudah di rata-rata kan.\n",
    "\n",
    "```{code}\n",
    "         date       NO2\n",
    "0  2023-10-01  0.000027\n",
    "1  2023-10-02  0.000024\n",
    "2  2023-10-03  0.000024\n",
    "3  2023-10-04  0.000021\n",
    "4  2023-10-05  0.000021\n",
    "<class 'pandas.core.frame.DataFrame'>\n",
    "RangeIndex: 731 entries, 0 to 730\n",
    "Data columns (total 2 columns):\n",
    " #   Column  Non-Null Count  Dtype\n",
    "---  ------  --------------  -----\n",
    " 0   date    731 non-null    object\n",
    " 1   NO2     731 non-null    float64\n",
    "dtypes: float64(1), object(1)\n",
    "memory usage: 11.5+ KB\n",
    "```\n",
    "\n",
    "### e. Deteksi Outlier IQR\n",
    "\n",
    "Setelah kita mengisi missing value menggunakan metode Interpolasi Linear, selanjutnya kita akan mendeteksi Outlier menggunakan metode IQR.\n",
    "\n",
    "```{code}\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "df = pd.read_csv(\"no2_timeseries_interpolated.csv\")\n",
    "\n",
    "df['date'] = pd.to_datetime(df['date'])\n",
    "\n",
    "# Hitung IQR\n",
    "Q1 = df['NO2'].quantile(0.25)\n",
    "Q3 = df['NO2'].quantile(0.75)\n",
    "IQR = Q3 - Q1\n",
    "\n",
    "lower_bound = Q1 - 1.5 * IQR\n",
    "upper_bound = Q3 + 1.5 * IQR\n",
    "\n",
    "# Filter outlier\n",
    "outliers_iqr = df[(df['NO2'] < lower_bound) | (df['NO2'] > upper_bound)]\n",
    "\n",
    "print(\"Jumlah Outlier (IQR):\", len(outliers_iqr))\n",
    "print(outliers_iqr[['date', 'NO2']].head())\n",
    "\n",
    "output/terminal\n",
    "Jumlah Outlier (IQR): 14\n",
    "         date       NO2\n",
    "45 2023-11-15  0.000051\n",
    "46 2023-11-16  0.000044\n",
    "48 2023-11-18  0.000051\n",
    "67 2023-12-07  0.000047\n",
    "68 2023-12-08  0.000045\n",
    "```\n",
    "\n",
    "Untuk men-visualisasi outlier:\n",
    "\n",
    "```{code}\n",
    "# === Visualisasi ===\n",
    "plt.figure(figsize=(15,5))\n",
    "plt.plot(df['date'], df['NO2'], label=\"NO2\", linewidth=1)\n",
    "\n",
    "# Titik Outlier\n",
    "plt.scatter(outliers_iqr['date'], outliers_iqr['NO2'], \n",
    "            color='red', marker='o', label=\"Outliers\")\n",
    "\n",
    "# Garis batas atas & bawah\n",
    "plt.axhline(upper_bound, color='orange', linestyle='dashed', label=\"Upper Bound (IQR)\")\n",
    "plt.axhline(lower_bound, color='blue', linestyle='dashed', label=\"Lower Bound (IQR)\")\n",
    "\n",
    "plt.title(\"Deteksi Outlier Data NO2 (Metode IQR)\")\n",
    "plt.xlabel(\"Tanggal\")\n",
    "plt.ylabel(\"Kadar NO2\")\n",
    "plt.legend()\n",
    "plt.tight_layout()\n",
    "plt.xticks(\n",
    "    ticks=[df['date'].iloc[0], df['date'].iloc[-1]],\n",
    "    labels=[df['date'].iloc[0].strftime('%Y-%m-%d'),\n",
    "            df['date'].iloc[-1].strftime('%Y-%m-%d')]\n",
    ")\n",
    "plt.show()\n",
    "```\n",
    "\n",
    "![Teks alternatif](img/Figure_1.png)\n",
    "\n",
    "Setelah itu, kita akan menghapus data outlier. Karena data ini merupakan data Time Series, maka data outlier yang dihapus akan diisi kembali menggunakan Interpolasi Linear.\n",
    "\n",
    "```{code}\n",
    "# Tandai outlier menjadi NaN\n",
    "df['NO2_cleaned'] = df['NO2'].mask((df['NO2'] < lower_bound) | (df['NO2'] > upper_bound))\n",
    "\n",
    "print(\"Jumlah nilai yang dinyatakan sebagai outlier:\", df['NO2_cleaned'].isna().sum())\n",
    "\n",
    "# Interpolasi linear untuk mengisi kembali nilai outlier\n",
    "df['NO2_filled'] = df['NO2_cleaned'].interpolate(method='linear')\n",
    "\n",
    "# Jika masih tersisa NaN di ujung data, isi dengan forward/backward fill\n",
    "df['NO2_filled'] = df['NO2_filled'].bfill().ffill()\n",
    "# df['NO2_filled'] = df['NO2_filled'].fillna(method='bfill').fillna(method='ffill')\n",
    "\n",
    "print(\"Jumlah missing setelah interpolasi:\", df['NO2_filled'].isna().sum())\n",
    "```\n",
    "\n",
    "Visualisasi data setelah menghapus Outlier dan mengisi kembali menggunakan Interpolasi Linear:\n",
    "\n",
    "```{code}\n",
    "plt.figure(figsize=(15,5))\n",
    "# Plot data hasil interpolasi\n",
    "plt.plot(df['date'], df['NO2_filled'], label=\"NO2 (Interpolated)\", linewidth=1)\n",
    "# Tampilkan hanya tanggal awal dan akhir di sumbu X\n",
    "plt.xticks(\n",
    "    ticks=[df['date'].iloc[0], df['date'].iloc[-1]],\n",
    "    labels=[df['date'].iloc[0].strftime('%Y-%m-%d'),\n",
    "            df['date'].iloc[-1].strftime('%Y-%m-%d')]\n",
    ")\n",
    "plt.title(\"Plot Data NO2 Setelah Outlier Removal & Interpolasi\")\n",
    "plt.xlabel(\"Tanggal\")\n",
    "plt.ylabel(\"Kadar NO2\")\n",
    "plt.legend()\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "```\n",
    "\n",
    "![Teks alternatif](img/Figure_2.png)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "ceed6c4b",
   "metadata": {},
   "source": [
    "## 3. Modeling menggunakan KNN Regression\n",
    "\n",
    "Dengan data Time Series kadar NO2 harian di daerah Bangkalan, kita akan memprediksi kadar NO2 satu hari yang akan datang. Sekarang kita akan ubah data, mencoba mencari korelasi antara 1 hari dengan 4 hari sebelumnya. Kita juga akan membandingkan apakah semakin banyak hari sebelumnya, model akan lebih bagus?\n",
    "\n",
    "### a. Uji Korelasi Data\n",
    "\n",
    "Sebelum masuk ke modeling, data kita merupakan data unsupervised yang berarti tidak ada label. Kita ubah data menjadi supervised lalu uji korelasi terhadap label (t). Fitur-fitur nya merupakan 30 hari sebelum (t-30, t-29, ... t-1) dan label (t).\n",
    "\n",
    "```{code}\n",
    "import pandas as pd\n",
    "\n",
    "def create_supervised(data, n_lag=4):\n",
    "    df_supervised = pd.DataFrame()\n",
    "    \n",
    "    # Membuat fitur t-4 sampai t-1\n",
    "    for i in range(n_lag, 0, -1):\n",
    "        df_supervised[f'NO2(t-{i})'] = data.shift(i)\n",
    "    \n",
    "    # Label hari H\n",
    "    df_supervised['NO2(t)'] = data\n",
    "    \n",
    "    # Hapus baris yang masih mengandung NaN akibat shift\n",
    "    df_supervised.dropna(inplace=True)\n",
    "    \n",
    "    return df_supervised\n",
    "\n",
    "# contoh penggunaan\n",
    "supervised_df30 = create_supervised(df['NO2_scaled'], n_lag=30)\n",
    "\n",
    "# Ambil semua lag dan kolom target\n",
    "lag_cols = supervised_df30.drop(columns=\"NO2(t)\").columns\n",
    "correlations = supervised_df30[lag_cols].corrwith(supervised_df30['NO2(t)'])\n",
    "\n",
    "# Tampilkan nilai korelasi\n",
    "print(correlations)\n",
    "\n",
    "output/terminal\n",
    "NO2(t-30)    0.442365\n",
    "NO2(t-29)    0.454480\n",
    "NO2(t-28)    0.475354\n",
    "NO2(t-27)    0.411464\n",
    "NO2(t-26)    0.381559\n",
    "NO2(t-25)    0.368824\n",
    "NO2(t-24)    0.353114\n",
    "NO2(t-23)    0.364938\n",
    "NO2(t-22)    0.372437\n",
    "NO2(t-21)    0.380476\n",
    "NO2(t-20)    0.350856\n",
    "NO2(t-19)    0.342492\n",
    "NO2(t-18)    0.312603\n",
    "NO2(t-17)    0.283336\n",
    "NO2(t-16)    0.288346\n",
    "NO2(t-15)    0.292171\n",
    "NO2(t-14)    0.311974\n",
    "NO2(t-13)    0.327142\n",
    "NO2(t-12)    0.341764\n",
    "NO2(t-11)    0.374090\n",
    "NO2(t-10)    0.397377\n",
    "NO2(t-9)     0.419258\n",
    "NO2(t-8)     0.455909\n",
    "NO2(t-7)     0.462456\n",
    "NO2(t-6)     0.460161\n",
    "NO2(t-5)     0.491515\n",
    "NO2(t-4)     0.523820\n",
    "NO2(t-3)     0.593839\n",
    "NO2(t-2)     0.675955\n",
    "NO2(t-1)     0.796441\n",
    "```\n",
    "\n",
    "Skala nilai uji korelasi itu dari -1 sampai 1, namun kita ambil nilai uji korelasi yang terbaik yaitu lebih dari 0.5 yaitu fitur t-1 sampai t-4.\n",
    "\n",
    "### b. Normalisasi Data\n",
    "\n",
    "karena kita menggunakan model KNN Regression, maka perlu normalisasi data menggunakan min-max Scaler.\n",
    "\n",
    "```{code}\n",
    "from sklearn.preprocessing import MinMaxScaler\n",
    "import pandas as pd\n",
    "\n",
    "scaler = MinMaxScaler()\n",
    "\n",
    "df['NO2_scaled'] = scaler.fit_transform(df[['NO2']])\n",
    "```\n",
    "\n",
    "Maka data akan di-normalisasi 0-1.\n",
    "\n",
    "```{code}\n",
    "output/terminal\n",
    "         date       NO2  NO2_scaled\n",
    "0  2023-10-01  0.000027    0.238203\n",
    "1  2023-10-02  0.000024    0.192840\n",
    "2  2023-10-03  0.000024    0.196854\n",
    "3  2023-10-04  0.000021    0.149560\n",
    "4  2023-10-05  0.000021    0.154247\n",
    "<class 'pandas.core.frame.DataFrame'>\n",
    "RangeIndex: 731 entries, 0 to 730\n",
    "Data columns (total 3 columns):\n",
    " #   Column      Non-Null Count  Dtype\n",
    "---  ------      --------------  -----\n",
    " 0   date        731 non-null    object\n",
    " 1   NO2         731 non-null    float64\n",
    " 2   NO2_scaled  731 non-null    float64\n",
    "dtypes: float64(2), object(1)\n",
    "memory usage: 17.3+ KB\n",
    "```\n",
    "\n",
    "### c. Mengubah Data\n",
    "\n",
    "Sekarang saya ingin mengubah data dari sebelumnya hanya 2 fitru menjadi 4 hari sebelum yang terdapat 5 fitur (t-4, t-3, t-2, t-1, dan t sebagai label) karena dari uji korelasi, keempat fitur tersebut (t-1 sampai t-4) merupakan nilai uji korelasi terbaik (lebih dari 0.5). Saya juga membuat data 10 hari sebelum untuk membandingkan apakah semakin banyak hari sebelum, semakin baik pula modelnya?\n",
    "\n",
    "```{code}\n",
    "supervised_df = create_supervised(df['NO2_scaled'], n_lag=4)\n",
    "\n",
    "print(supervised_df)\n",
    "print(supervised_df.shape)\n",
    "```\n",
    "\n",
    "```{code}\n",
    "output/terminal\n",
    "     NO2(t-4)  NO2(t-3)  NO2(t-2)  NO2(t-1)    NO2(t)\n",
    "4    0.238203  0.192840  0.196854  0.149560  0.154247\n",
    "5    0.192840  0.196854  0.149560  0.154247  0.185625\n",
    "6    0.196854  0.149560  0.154247  0.185625  0.152010\n",
    "7    0.149560  0.154247  0.185625  0.152010  0.149143\n",
    "8    0.154247  0.185625  0.152010  0.149143  0.159907\n",
    "..        ...       ...       ...       ...       ...\n",
    "726  0.123092  0.325742  0.372653  0.145997  0.094458\n",
    "727  0.325742  0.372653  0.145997  0.094458  0.089599\n",
    "728  0.372653  0.145997  0.094458  0.089599  0.000000\n",
    "729  0.145997  0.094458  0.089599  0.000000  0.014405\n",
    "730  0.094458  0.089599  0.000000  0.014405  0.014405\n",
    "[727 rows x 5 columns]\n",
    "(727, 5)\n",
    "```\n",
    "\n",
    "Untuk membuat data 10 hari sebelum tinggal tambah code dibawah (ubah parameter n_lag).\n",
    "\n",
    "```{code}\n",
    "supervised_df10 = create_supervised(df['NO2_scaled'], n_lag=10)\n",
    "\n",
    "print(supervised_df10)\n",
    "print(supervised_df10.shape)\n",
    "```\n",
    "\n",
    "```{code}\n",
    "output/terminal\n",
    "     NO2(t-10)  NO2(t-9)  NO2(t-8)  NO2(t-7)  NO2(t-6)  NO2(t-5)  NO2(t-4)  NO2(t-3)  NO2(t-2)  NO2(t-1)    NO2(t)\n",
    "10    0.238203  0.192840  0.196854  0.149560  0.154247  0.185625  0.152010  0.149143  0.159907  0.242292  0.214105\n",
    "11    0.192840  0.196854  0.149560  0.154247  0.185625  0.152010  0.149143  0.159907  0.242292  0.214105  0.166780\n",
    "12    0.196854  0.149560  0.154247  0.185625  0.152010  0.149143  0.159907  0.242292  0.214105  0.166780  0.127252\n",
    "13    0.149560  0.154247  0.185625  0.152010  0.149143  0.159907  0.242292  0.214105  0.166780  0.127252  0.083753\n",
    "14    0.154247  0.185625  0.152010  0.149143  0.159907  0.242292  0.214105  0.166780  0.127252  0.083753  0.091532\n",
    "..         ...       ...       ...       ...       ...       ...       ...       ...       ...       ...       ...\n",
    "726   0.161874  0.128849  0.095824  0.062799  0.038033  0.059606  0.123092  0.325742  0.372653  0.145997  0.094458\n",
    "727   0.128849  0.095824  0.062799  0.038033  0.059606  0.123092  0.325742  0.372653  0.145997  0.094458  0.089599\n",
    "728   0.095824  0.062799  0.038033  0.059606  0.123092  0.325742  0.372653  0.145997  0.094458  0.089599  0.000000\n",
    "729   0.062799  0.038033  0.059606  0.123092  0.325742  0.372653  0.145997  0.094458  0.089599  0.000000  0.014405\n",
    "730   0.038033  0.059606  0.123092  0.325742  0.372653  0.145997  0.094458  0.089599  0.000000  0.014405  0.014405\n",
    "[721 rows x 11 columns]\n",
    "(721, 11)\n",
    "```\n",
    "\n",
    "### d. Modeling dan Evaluation\n",
    "\n",
    "Sekarang dari 2 data yang sudah kita rubah, kita train menggunakan model KNN Regression.\n",
    "\n",
    "```{code}\n",
    "from sklearn.neighbors import KNeighborsRegressor\n",
    "from sklearn.model_selection import train_test_split\n",
    "from sklearn.metrics import mean_squared_error, r2_score\n",
    "import numpy as np\n",
    "\n",
    "def MAPE(y_true, y_pred):\n",
    "    y_true, y_pred = np.array(y_true), np.array(y_pred)\n",
    "    # Hindari pembagian dengan nol\n",
    "    nonzero = y_true != 0\n",
    "    return np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100\n",
    "\n",
    "def train_knn(df_supervised, model_name=\"\"):\n",
    "    # Pisahkan fitur & label\n",
    "    X = df_supervised.drop(columns=['NO2(t)']).values\n",
    "    y = df_supervised['NO2(t)'].values\n",
    "\n",
    "    # Split data 80/20\n",
    "    X_train, X_test, y_train, y_test = train_test_split(\n",
    "        X, y, test_size=0.2, shuffle=False\n",
    "    )\n",
    "\n",
    "    # Model KNN\n",
    "    knn = KNeighborsRegressor(n_neighbors=5)\n",
    "    knn.fit(X_train, y_train)\n",
    "\n",
    "    # Prediksi\n",
    "    y_pred = knn.predict(X_test)\n",
    "\n",
    "    # Evaluasi\n",
    "    mse = mean_squared_error(y_test, y_pred)\n",
    "    rmse = np.sqrt(mse)\n",
    "    r2 = r2_score(y_test, y_pred)\n",
    "    mape = MAPE(y_test, y_pred)\n",
    "\n",
    "    print(f\"\\n=== {model_name} ===\")\n",
    "    print(f\"Train Size: {len(X_train)} — Test Size: {len(X_test)}\")\n",
    "    print(f\"RMSE: {rmse:.6f}\")\n",
    "    print(f\"R² Score: {r2:.4f}\")\n",
    "    print(f\"MAPE: {mape:.4f}%\")\n",
    "\n",
    "    return knn, y_test, y_pred\n",
    "\n",
    "\n",
    "# Train model untuk 4 hari sebelumnya\n",
    "knn_4, y_test_4, y_pred_4 = train_knn(supervised_df, \"KNN - 4 Hari Sebelumnya\")\n",
    "\n",
    "# Train model untuk 10 hari sebelumnya\n",
    "knn_10, y_test_10, y_pred_10 = train_knn(supervised_df10, \"KNN - 10 Hari Sebelumnya\")\n",
    "```\n",
    "\n",
    "```{code}\n",
    "output/terminal\n",
    "=== KNN - 4 Hari Sebelumnya ===\n",
    "Train Size: 581 — Test Size: 146\n",
    "RMSE: 0.065436\n",
    "R² Score: 0.1395\n",
    "MAPE: 61.0780%\n",
    "\n",
    "=== KNN - 10 Hari Sebelumnya ===\n",
    "Train Size: 576 — Test Size: 145\n",
    "RMSE: 0.067567\n",
    "R² Score: 0.0886\n",
    "MAPE: 64.6611%\n",
    "```\n",
    "\n",
    "Berdasarkan hasil arkurasi diatas menjunjukkan bahwa lebih banyak hari sebelumnya maka model semakin bagus. Kita coba gunakan data 30 hari sebelumnya juga untuk melihat apakah semakin banyak hari sebelumnya, model semakin baik?\n",
    "\n",
    "```{code}\n",
    "knn_30, y_test_30, y_pred_30 = train_knn(supervised_df30, \"KNN - 30 Hari Sebelumnya\")\n",
    "\n",
    "output/terminal\n",
    "=== KNN - 30 Hari Sebelumnya ===\n",
    "Train Size: 560 — Test Size: 141\n",
    "RMSE: 0.074803\n",
    "R² Score: -0.0875\n",
    "MAPE: 72.2295%\n",
    "```\n",
    "\n",
    "### e. Plotting\n",
    "\n",
    "Plotting untuk visualisasi grafik antara label dan prediksi dari kedua data diatas.\n",
    "\n",
    "4 hari sebelum:\n",
    "\n",
    "```{code}\n",
    "import matplotlib.pyplot as plt\n",
    "import numpy as np\n",
    "\n",
    "plt.figure()\n",
    "plt.plot(np.arange(len(y_test_4)), y_test_4, label=\"Actual\")\n",
    "plt.plot(np.arange(len(y_pred_4)), y_pred_4, label=\"Predicted\")\n",
    "plt.title(\"KNN Regression - 4 Hari Sebelumnya\")\n",
    "plt.xlabel(\"Sample Index\")\n",
    "plt.ylabel(\"NO2 Value\")\n",
    "plt.legend()\n",
    "plt.show()\n",
    "```\n",
    "\n",
    "![Teks alternatif](img/Screenshot%202025-10-25%20163935.png)\n",
    "\n",
    "10 hari sebelum:\n",
    "\n",
    "```{code}\n",
    "plt.figure()\n",
    "plt.plot(np.arange(len(y_test_10)), y_test_10, label=\"Actual\")\n",
    "plt.plot(np.arange(len(y_pred_10)), y_pred_10, label=\"Predicted\")\n",
    "plt.title(\"KNN Regression - 10 Hari Sebelumnya\")\n",
    "plt.xlabel(\"Sample Index\")\n",
    "plt.ylabel(\"NO2 Value\")\n",
    "plt.legend()\n",
    "plt.show()\n",
    "```\n",
    "\n",
    "![Teks alternatif](img/Screenshot%202025-10-25%20164003.png)\n",
    "\n",
    "30 hari sebelum:\n",
    "\n",
    "```{code}\n",
    "plt.figure()\n",
    "plt.plot(np.arange(len(y_test_30)), y_test_30, label=\"Actual\")\n",
    "plt.plot(np.arange(len(y_pred_30)), y_pred_30, label=\"Predicted\")\n",
    "plt.title(\"KNN Regression - 30 Hari Sebelumnya\")\n",
    "plt.xlabel(\"Sample Index\")\n",
    "plt.ylabel(\"NO2 Value\")\n",
    "plt.legend()\n",
    "plt.show()\n",
    "```\n",
    "\n",
    "![Teks alternatif](img/Screenshot%202025-10-27%20115826.png)\n",
    "\n",
    "Hasil evaluasi model KNN Regression menunjukkan bahwa peningkatan jumlah fitur historis (lag) tidak serta merta meningkatkan performa prediksi. Pada model dengan 4 hari sebelumnya, nilai RMSE paling kecil dan R² masih positif sehingga model mampu menjelaskan sebagian kecil variabilitas data target. Namun, ketika jumlah lag ditambah menjadi 10 dan 30 hari sebelumnya, performa model justru menurun yang ditunjukkan oleh meningkatnya nilai RMSE dan MAPE, serta penurunan nilai R² hingga bernilai negatif pada lag 30. Nilai MAPE yang cukup tinggi pada seluruh model (lebih dari 60%) juga mengindikasikan bahwa akurasi prediksi masih rendah dan terdapat deviasi besar antara nilai prediksi dan nilai aktual. Secara keseluruhan, model KNN tidak memberikan performa yang baik pada data ini, dan penambahan fitur historis justru menyebabkan overfitting serta menurunkan kemampuan generalisasi model. Oleh karena itu, diperlukan pemilihan model lain atau peningkatan strategi preprocessing untuk memperoleh hasil prediksi yang lebih baik."
   ]
  }
 ],
 "metadata": {
  "language_info": {
   "name": "python"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}